# Advanced usage guide

This ipython notebook provides an advanced usage guide for the library. 

## 1. Agent custom setup

Instead of using the `setup_agent` function, you can create a custom agent setup by directly instantiating the `Agent` class and configuring it with dependencies. 

Below example creates an agent with a dummy display and registers a simple addition function to the agent's toolbox. 

The agent execution always returns a `Result` object, which can be unwrapped to get the final answer.

In [ ]:
from xun import Agent, ToolBox, NullDisplay

def add(a: int, b: int) -> int:
    return a + b

agent = Agent(
    display=NullDisplay(), 
    toolbox=ToolBox().register(add)
    )
answer = agent.instruct("What is 2 + 3?").execute()

print(answer.unwrap())



2 + 3 equals 5.


## 2. The event-driven display interface
The agent always accepts a `DisplayAbstract` object as a display interface for IO.

The framework provides two display implementations: `Display` (default) and `NullDisplay`, 
where `Display` is a simple console display and `NullDisplay` is a dummy display that does nothing.

To define a custom display, you can implement the `DisplayAbstract` interface with only two methods: 
- `on_event`: called when an event occurs, such as a tool message. 
- `get_confirm`: called when the agent needs a confirmation from the user.

For example, below we define a custom display that prints the raw event to the console.

In [ ]:
from xun import DisplayAbstract

class RawDisplay(DisplayAbstract):
    def on_event(self, event):
        content = event.event
        print(f"Event type: {type(content).__name__}, content: {content}")

    def get_confirm(self, *args, **kwargs) -> bool:
        raise NotImplementedError("For demonstration purposes, this method is not implemented.")

agent = Agent(display=RawDisplay())
agent.toolbox.register(add)

_ = agent.instruct("What is 2 + 3?").execute()

Event type: ModelWorkingEvent, content: model_call_id='af584285-a7ab-4c61-9ffd-c741e845a49e' remaining_iterations=64
Event type: ToolCallEvent, content: tool_call_id='chatcmpl-tool-b7a0c098fde9c582' tool_name='add' args={'a': 2, 'b': 3}
Event type: ToolResultEvent, content: tool_call_id='chatcmpl-tool-b7a0c098fde9c582' result=5
Event type: ModelWorkingEvent, content: model_call_id='0e43c9d5-31a8-4c36-b0fa-308760784876' remaining_iterations=63
Event type: ModelMessageEvent, content: model_call_id='0e43c9d5-31a8-4c36-b0fa-308760784876' content='\n\n2 + 3 equals 5.'


## 3. Tool attributes

We can attach metadata to a tool function, for example, to change the tool name.

In [ ]:
from xun import tool_attr, setup_agent

@tool_attr(name="MultiplyTool")
def multiply(a: int, b: int) -> int:
    return a * b

agent = setup_agent(tools=[multiply], display=NullDisplay())
agent.instruct("What is 2 * 3?").execute()

tool_name = agent.instruct("What tool did you use?").execute().unwrap()

print(f"Reply from agent: {tool_name}")

Reply from agent: 

I used the `MultiplyTool` to calculate the result.


## 4. Structured output

The framework support pydantic model validation for tool output. 
You can define a pydantic model and use it as the parameter for the execution. 

- There will be prompt injected to the message, informing the agent of the correct format. 
- The output will be automatically validated and deduced for type hinting. 

In [ ]:
from pydantic import BaseModel

class StoryModel(BaseModel):
    title: str
    content: str

story = setup_agent(display = NullDisplay()).instruct(
    "Write a short story within 100 words."
).execute(schema=StoryModel).unwrap()

print(f"Title: {story.title}")
print(f"Content: {story.content}")

Title: The Last Battery
Content: Elara held the fading light of the last AA battery. The air purifier sputtered, a dying breath in the silent bunker. Outside, the world was gray ash. Inside, dust motes danced in the weak beam. She placed the battery into her father's old music box. As the spring tightened, a faint, tinkling melody filled the room. For a moment, the gray world receded. She wasn't just surviving; she was remembering. The melody looped, a fragile tether to a sunlit past, before the battery finally went dark.


## 5. Tool execution context

While a tool is just a plain Python function without state, 
the framework provides an execution context for it.

Simply declare a `ToolCallContext` parameter in the tool function, and the framework will automatically inject the context object when the tool runs.

The context includes several built-in attributes, and you can also attach your own values when invoking the tool.

In [ ]:
from xun import ToolCallContext as Context
import datetime

def tool_with_context(context: Context[dict]):
    print("Tool called by: ", context.agent.name)
    context.value["tool_call_time"] = datetime.datetime.now().strftime("%H:%M:%S")

context_value = {"tool_call_time": "?"}
setup_agent(
    name = "ContextAgent",
    tools=[tool_with_context], 
    display=NullDisplay()
).instruct(
    "Call the only tool you have in hand."
).execute(context=context_value)

print(f"Context after tool execution: {context_value}")

Tool called by:  ContextAgent
Context after tool execution: {'tool_call_time': '19:24:36'}


## 6. Sub-agent spawning

It is easy to make agent spawnning a tool, just to create a new agent in the function to execute the tool call. 

As another approach, the framework provides a quick and generic way to quickly setup sub-agent spawning. 
By using the `ToolBox::with_subagent_provider` function, you register a generic sub-agent call tools to execute tasks.

This function support declare an agent-getter function, if its omitted, the framework will use the default agent setup to spawn a sub-agent (inheriting the parent agent's toolbox and display).

In [ ]:
import math

def sqrt(ctx: Context, a: float) -> float:
    print(f"Tool called by: {ctx.agent.name}")
    return math.sqrt(a)

def agent_getter(_):
    return setup_agent(display=NullDisplay(), tools=[sqrt])

agent = Agent(
    display = NullDisplay(), 
    toolbox = ToolBox().with_subagent_provider(agent_getter)
    )
result = agent.instruct("What is the result of square root of 114514? call a sub-agent to calculate it.").execute()

print(f"Result from sub-agent: {result.unwrap()}")

Tool called by: subagent
Result from sub-agent: 

The square root of 114514 is approximately **338.40**.

(Specifically, it is 338.39917257582056).


## 7. Lifecycle hooks

We can register hooks to the agent execution.
Currently, only supports tool call related hooks, may add more in the future.

In [7]:
agent = setup_agent(display=NullDisplay(), tools = [add])
agent.hooks.before_tool_call.add(lambda arg: print(arg.tool_calls))
agent.hooks.after_tool_call.add(lambda arg: print(arg.tool_results))

_ = agent.instruct("What is 2 + 3?").execute()

[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-95234972a49d5ffa', function=Function(arguments='{"a": 2, "b": 3}', name='add'), type='function')]
[('chatcmpl-tool-95234972a49d5ffa', <xun.types.Result object at 0x70aa894b1310>)]
